# 🌤️ CampingTN — Weather Analysis & Seasonal Safety Model
## Notebook 3: Climate Patterns, Risk Scoring & Seasonal Recommendations

Builds:
1. Weather risk classifier per region/season
2. Best-season recommender per site type
3. Temperature & humidity analysis across Tunisia
4. Export: `weather_risk_model.pkl`

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn joblib -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings; warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Ready')

In [ ]:
# Typical climate data for Tunisian regions
CLIMATE = {
    'Tunis':     {'SPRING':(22,68),'SUMMER':(32,65),'AUTUMN':(20,70),'WINTER':(13,78)},
    'Bizerte':   {'SPRING':(20,72),'SUMMER':(30,68),'AUTUMN':(19,74),'WINTER':(12,82)},
    'Sousse':    {'SPRING':(22,68),'SUMMER':(32,65),'AUTUMN':(21,70),'WINTER':(13,76)},
    'Sfax':      {'SPRING':(23,65),'SUMMER':(33,62),'AUTUMN':(22,67),'WINTER':(14,74)},
    'Kasserine': {'SPRING':(20,58),'SUMMER':(35,40),'AUTUMN':(20,55),'WINTER':(9,65)},
    'Gabès':     {'SPRING':(24,55),'SUMMER':(37,45),'AUTUMN':(25,52),'WINTER':(15,62)},
    'Tozeur':    {'SPRING':(27,30),'SUMMER':(43,18),'AUTUMN':(26,28),'WINTER':(16,40)},
    'Kébili':    {'SPRING':(28,25),'SUMMER':(44,16),'AUTUMN':(27,24),'WINTER':(16,38)},
    'Jendouba':  {'SPRING':(21,70),'SUMMER':(31,60),'AUTUMN':(20,72),'WINTER':(10,82)},
    'Médenine':  {'SPRING':(25,52),'SUMMER':(38,42),'AUTUMN':(26,50),'WINTER':(16,58)},
    'Tataouine': {'SPRING':(26,35),'SUMMER':(42,20),'AUTUMN':(25,32),'WINTER':(15,42)},
    'Nabeul':    {'SPRING':(21,70),'SUMMER':(31,68),'AUTUMN':(20,72),'WINTER':(13,78)},
    'Monastir':  {'SPRING':(22,67),'SUMMER':(32,64),'AUTUMN':(21,69),'WINTER':(13,76)},
}

# Risk levels: 0=LOW, 1=MEDIUM, 2=HIGH, 3=EXTREME
def compute_risk(temp, humidity, site_type, season):
    risk = 0
    if site_type == 'DESERT':
        if season == 'SUMMER': risk = 3  # extreme
        elif temp > 35: risk = 2
        elif temp > 28: risk = 1
    elif site_type == 'COASTAL':
        if season == 'WINTER' and humidity > 80: risk = 1
        elif season == 'SUMMER' and temp > 38: risk = 2
        else: risk = 0
    elif site_type == 'FOREST':
        if season == 'SUMMER' and temp > 35: risk = 2  # fire risk
        elif season == 'WINTER' and temp < 8: risk = 1
        else: risk = 0
    return risk

# Generate dataset
rows = []
for gov, seasons in CLIMATE.items():
    for season, (temp_avg, hum_avg) in seasons.items():
        for site in ['DESERT','COASTAL','FOREST']:
            for _ in range(40):
                t = temp_avg + np.random.normal(0, 3)
                h = hum_avg + np.random.normal(0, 8)
                risk = compute_risk(t, h, site, season)
                rows.append({'governorate':gov,'season':season,'site_type':site,
                             'temperature':round(t,1),'humidity':round(max(5,min(100,h)),1),
                             'risk_level':risk})

df_weather = pd.DataFrame(rows)
print(f'Weather dataset: {df_weather.shape}')
print(df_weather['risk_level'].value_counts())

In [ ]:
# Visualize climate across Tunisia
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Tunisia Climate Analysis for Camping Safety', fontsize=15, fontweight='bold')

# Avg temp by region and season
temp_data = df_weather.groupby(['governorate','season'])['temperature'].mean().unstack()
temp_data.plot(kind='bar', ax=axes[0,0],
               color=['#2d6a4f','#e63946','#e9c46a','#0077b6'])
axes[0,0].set_title('Average Temperature by Region & Season')
axes[0,0].set_ylabel('Temperature (°C)')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].legend(title='Season', fontsize=8)

# Risk distribution
risk_labels = {0:'Low',1:'Medium',2:'High',3:'Extreme'}
risk_counts = df_weather['risk_level'].map(risk_labels).value_counts()
axes[0,1].pie(risk_counts.values, labels=risk_counts.index,
              autopct='%1.1f%%', colors=['#2d6a4f','#e9c46a','#f4a261','#e63946'],
              startangle=90)
axes[0,1].set_title('Risk Level Distribution')

# Temperature heatmap
pivot = df_weather.groupby(['governorate','season'])['temperature'].mean().unstack()
sns.heatmap(pivot, ax=axes[1,0], cmap='RdYlGn_r', annot=True, fmt='.0f',
            linewidths=0.5, cbar_kws={'label':'°C'})
axes[1,0].set_title('Temperature Heatmap (°C)')

# Best season per site type
best = df_weather.groupby(['site_type','season'])['risk_level'].mean().unstack()
best.plot(kind='bar', ax=axes[1,1],
          color=['#2d6a4f','#e63946','#e9c46a','#0077b6'])
axes[1,1].set_title('Average Risk Level by Site Type & Season\n(Lower = Safer)')
axes[1,1].set_ylabel('Avg Risk Level')
axes[1,1].tick_params(axis='x', rotation=0)
axes[1,1].legend(title='Season', fontsize=8)

plt.tight_layout()
plt.savefig('weather_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Train risk classifier
le_gov2 = LabelEncoder().fit(df_weather['governorate'])
le_season2 = LabelEncoder().fit(df_weather['season'])
le_site2 = LabelEncoder().fit(df_weather['site_type'])

X_w = pd.DataFrame({
    'governorate': le_gov2.transform(df_weather['governorate']),
    'season': le_season2.transform(df_weather['season']),
    'site_type': le_site2.transform(df_weather['site_type']),
    'temperature': df_weather['temperature'],
    'humidity': df_weather['humidity'],
})
y_w = df_weather['risk_level']

X_tr, X_te, y_tr, y_te = train_test_split(X_w, y_w, test_size=0.2, random_state=42)
risk_clf = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42, n_jobs=-1)
risk_clf.fit(X_tr, y_tr)
print(classification_report(y_te, risk_clf.predict(X_te),
                             target_names=['Low','Medium','High','Extreme']))

weather_pkg = {
    'classifier': risk_clf,
    'le_gov': le_gov2, 'le_season': le_season2, 'le_site': le_site2,
    'climate_data': CLIMATE,
    'risk_labels': {0:'LOW',1:'MEDIUM',2:'HIGH',3:'EXTREME'},
    'version': '1.0'
}
joblib.dump(weather_pkg, 'weather_risk_model.pkl', compress=3)
print('✅ Saved: weather_risk_model.pkl')